### Figgie Random Sampling to understand nature of game

In [1]:
from enum import Enum

import numpy as np
import numpy.random as random
import matplotlib.pyplot as plt
import time

In [43]:
NSAMPLES = 1e9
SUIT_SIZES = [10, 10, 12, 8]
COMMON_IDX = SUIT_SIZES.index(12)
NONCOMMON_SUIT_SIZES = [10, 10, 8]

class SUIT_CLASS(Enum):
    GOAL = 0
    NORMAL = 1
    COMMON = 2

class Suit():
    def __init__(self, name, color, suit_class_idx: int):
        self._name: str = name
        self._color: str = color
        self._suit_class: SUIT_CLASS = SUIT_CLASS(suit_class_idx)
        self._suit_size: int = 0

    def __str__(self):
        return (f"{self._name}\n  {self._color}\n  {self._suit_class}, {self._suit_size}")
    
    def get_name(self):
        return self._name
    
    def get_color(self) -> str:
        if self._color == "Black":
            return "Black"
        elif self._color == "Red":
            return "Red"
        else:
            return "Invalid card color"
    
    def get_color_int(self) -> int:
        if self._color == "Black":
            return 0
        else:
            return 1
    
    def get_suit_class(self) -> SUIT_CLASS:
        return self._suit_class
    
    def get_suit_class_int(self) -> int:
        return self._suit_class.value
    
    def get_suit_size(self) -> int:
        return self._suit_size

    def set_suit_class(self, class_int):
        self._suit_class = SUIT_CLASS(class_int)

    def set_suit_size(self, size_int):
        if (self._suit_class == SUIT_CLASS.COMMON):
            self._suit_size = SUIT_SIZES[COMMON_IDX]
        elif (size_int in NONCOMMON_SUIT_SIZES):
            self._suit_size = size_int
        else:
            raise ValueError(f"Disallowed suit size: {size_int}")

In [178]:
Suits = [Suit("Spades", "Black", 1), 
         Suit("Clubs", "Black", 1), 
         Suit("Hearts", "Red", 1), 
         Suit("Diamonds", "Red", 1)]
available_classes = [SUIT_CLASS.COMMON, SUIT_CLASS.NORMAL, SUIT_CLASS.NORMAL, SUIT_CLASS.GOAL]
noncommon_size_cpy = NONCOMMON_SUIT_SIZES.copy()
Suit_cpy = Suits.copy()

In [179]:
"""
Brainstorming random selection algorithm:

1. Select a Suit (Spade, Club, Heart, Diamond)
2. If a special color has been assigned:
    2a. If the special color matches the current suit's color:
        2a.i. If the common suit has been assigned: set current suit to goal
        2a.ii. else: set current suit to common
    2b. Else: set current suit class to Normal
2c. Else (color hasn't been assigned): set suit class (Common, Goal, Normal)
    2ci. If class is Common: set special color = suit.color, assign common suit, set suit size = 12
    2cii. Else if class is Goal: set special color = suit.color, assign goal suit
3. If current suit isn't common: set suit size (10, 10, 8)
4. delete current suit from suit_copy, current class from available_classes, current size from noncommon_size_cpy
"""

"\nBrainstorming random selection algorithm:\n\n1. Select a Suit (Spade, Club, Heart, Diamond)\n2. If a special color has been assigned:\n    2a. If the special color matches the current suit's color:\n        2a.i. If the common suit has been assigned: set current suit to goal\n        2a.ii. else: set current suit to common\n    2b. Else: set current suit class to Normal\n2c. Else (color hasn't been assigned): set suit class (Common, Goal, Normal)\n    2ci. If class is Common: set special color = suit.color, assign common suit, set suit size = 12\n    2cii. Else if class is Goal: set special color = suit.color, assign goal suit\n3. If current suit isn't common: set suit size (10, 10, 8)\n4. delete current suit from suit_copy, current class from available_classes, current size from noncommon_size_cpy\n"

In [180]:
# Step 1: Deck Distribution

Suits = [Suit("Spades", "Black", 1), 
         Suit("Clubs", "Black", 1), 
         Suit("Hearts", "Red", 1), 
         Suit("Diamonds", "Red", 1)]
available_classes = [SUIT_CLASS.COMMON, SUIT_CLASS.NORMAL, SUIT_CLASS.NORMAL, SUIT_CLASS.GOAL]
noncommon_size_cpy = NONCOMMON_SUIT_SIZES.copy()
Suit_cpy = Suits.copy()

time_start = time.perf_counter()
special_color = None
common_suit = None
goal_suit = None
for suit in Suits:
    rand_suit = random.choice(Suit_cpy)
    rand_suit_idx = Suits.index(rand_suit)

    if special_color:
        if (special_color == rand_suit.get_color()):
            if (common_suit): 
                Suits[rand_suit_idx].set_suit_class(SUIT_CLASS.GOAL.value)
            else:
                Suits[rand_suit_idx].set_suit_class(SUIT_CLASS.COMMON.value)
        else:
            Suits[rand_suit_idx].set_suit_class(SUIT_CLASS.NORMAL.value)
    else:
        rand_class = random.choice(available_classes)
        Suits[rand_suit_idx].set_suit_class(rand_class.value)
        if (rand_class == SUIT_CLASS.COMMON):
            special_color = Suits[rand_suit_idx].get_color()
            common_suit = Suits[rand_suit_idx].get_name()
        elif (rand_class == SUIT_CLASS.GOAL):
            special_color = Suits[rand_suit_idx].get_color()
            goal_suit = Suits[rand_suit_idx].get_name()

    if (Suits[rand_suit_idx].get_suit_class() != SUIT_CLASS.COMMON):
        rand_size = random.choice(noncommon_size_cpy)
        Suits[rand_suit_idx].set_suit_size(rand_size)
        noncommon_size_cpy.remove(rand_size)
    else:
        Suits[rand_suit_idx].set_suit_size(12)
    
    Suit_cpy.remove(Suits[rand_suit_idx])
    available_classes.remove(Suits[rand_suit_idx].get_suit_class())

Suit_cpy = Suits.copy()
available_classes = [SUIT_CLASS.COMMON, SUIT_CLASS.NORMAL, SUIT_CLASS.NORMAL, SUIT_CLASS.GOAL]
suit_size_cpy = SUIT_SIZES.copy()

time_elapsed = time.perf_counter() - time_start
print(time_elapsed)
for suit in Suits: print(suit)

0.0008462779987894464
Spades
  Black
  SUIT_CLASS.COMMON, 12
Clubs
  Black
  SUIT_CLASS.GOAL, 8
Hearts
  Red
  SUIT_CLASS.NORMAL, 10
Diamonds
  Red
  SUIT_CLASS.NORMAL, 10


In [181]:
# Step 2: Individual Hand Distribution

num_cards = {suit.get_name(): suit.get_suit_size() for suit in Suits}
suit_names = list(num_cards.keys())
suit_sizes = np.array(list(num_cards.values()))

probs = suit_sizes / suit_sizes.sum()
rng = random.default_rng()
selections = rng.choice(suit_names, size=8, replace=True, p=probs)

indiv_suits, indiv_cts = np.unique(selections, return_counts=True)
indiv_hand = {str(indiv_suits[i]): int(indiv_cts[i]) for i in range(len(indiv_suits))}

print(indiv_hand)

{'Clubs': 1, 'Diamonds': 3, 'Hearts': 3, 'Spades': 1}


In [53]:
# step 3: Abstract to a class for sampling

class Figgie_Deck():
    def __init__(self):
        self._suits = [Suit("Spades", "Black", 1), 
                       Suit("Clubs", "Black", 1), 
                       Suit("Hearts", "Red", 1), 
                       Suit("Diamonds", "Red", 1)]
        
        self._available_classes = [SUIT_CLASS.GOAL, 
                                   SUIT_CLASS.NORMAL, 
                                   SUIT_CLASS.NORMAL, 
                                   SUIT_CLASS.COMMON]
        self._noncommon_sizes = [10, 10, 8]
        self._special_color = None
        self._common_suit = None
        self._goal_suit = None
    
    def __str__(self):
        deck_str = ""
        for suit in self._suits:
            deck_str += suit
            deck_str += "\n"
        return deck_str
    
    def get_special_color(self):
        # must call self.shuffle_deck() at least once before calling this function
        # to get something other than None
        return self._special_color

    def get_common_suit(self):
        # must call self.shuffle_deck() at least once before calling this function
        # to get something other than None
        return self._common_suit

    def get_goal_suit(self):
        # must call self.shuffle_deck() at least once before calling this function
        # to get something other than None
        return self._goal_color

    def shuffle_deck(self) -> float:
        # randomizes deck distribution according to Figgie rules
        # returns runtime

        self.deck_reset()

        time_start = time.perf_counter()

        noncommon_size_cpy = self._noncommon_sizes.copy()

        common_suit = random.choice(self._suits)
        common_idx = self._suits.index(common_suit)
        self._suits[common_idx].set_suit_class(SUIT_CLASS.COMMON)
        self._suits[common_idx].set_suit_size(SUIT_SIZES[COMMON_IDX])
        for suit in self._suits:
            if suit == common_suit: continue
            rand_size = random.choice(noncommon_size_cpy)
            if (suit.get_color() == common_suit.get_color()):
                if (suit != common_suit):
                    goal_suit = suit
                    goal_idx = self._suits.index(goal_suit)
                    self._suits[goal_idx].set_suit_class(SUIT_CLASS.GOAL)
                    self._suits[goal_idx].set_suit_size(rand_size)
            else:
                self._suits[self._suits.index(suit)].set_suit_class(SUIT_CLASS.NORMAL)
                self._suits[self._suits.index(suit)].set_suit_size(rand_size)

            noncommon_size_cpy.remove(rand_size)                  

        time_elapsed = time.perf_counter() - time_start
        print(f"Runtime: {time_elapsed*1e3:.2f}ms")
        return float(time_elapsed)

    def deck_reset(self):
        self._suits = [Suit("Spades", "Black", 1), 
                       Suit("Clubs", "Black", 1), 
                       Suit("Hearts", "Red", 1), 
                       Suit("Diamonds", "Red", 1)]

        self._noncommon_sizes = [10, 10, 8]
        self._special_color = None
        self._common_suit = None
        self._goal_suit = None

In [54]:
figgie_test1 = Figgie_Deck()
for i in range(100):
    figgie_test1.shuffle_deck()

Runtime: 0.69ms
Runtime: 0.10ms
Runtime: 1.11ms
Runtime: 0.38ms
Runtime: 0.28ms
Runtime: 0.67ms
Runtime: 0.30ms
Runtime: 0.23ms
Runtime: 0.38ms
Runtime: 0.26ms
Runtime: 0.37ms
Runtime: 0.34ms
Runtime: 0.28ms
Runtime: 0.32ms
Runtime: 0.17ms
Runtime: 0.31ms
Runtime: 0.28ms
Runtime: 0.63ms
Runtime: 0.09ms
Runtime: 0.06ms
Runtime: 0.03ms
Runtime: 0.20ms
Runtime: 0.08ms
Runtime: 0.06ms
Runtime: 0.04ms
Runtime: 0.74ms
Runtime: 0.04ms
Runtime: 0.06ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 1.70ms
Runtime: 0.07ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 0.03ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 0.04ms
Runtime: 0.03ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.04ms
Runtime: 0.03ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime: 0.02ms
Runtime:

In [ ]:
# Step 4: Sampling Individual Decks